# NBA Player ROI Calculator — 2024-25 Season
## Based on: *A Framework for NBA ROI* (Lautier et al.) — Simplified Linear Model

This notebook computes a **Return on Investment (ROI)** for every NBA player in the 2024-25
regular season, comparing their on-court performance against their salary.

### Methodology — Linear Performance Model (No IRR)
This version retains the Lautier et al. variable list and coefficients but replaces the
logistic regression probability framework and IRR solve with a direct linear performance
score and a simple cash-flow ratio. The goal is to measure what actually happened, not
to model the probability of winning.

**Five-step process:**
1. **Performance score (Delta)** — Weighted sum of centered stats per game:
   `Delta_gm = sum(beta_j * (x_gm_j - x_bar_j))`. Same 24 Lautier variables and
   coefficients; no sigmoid or probability conversion.
2. **WRMS** — Wealth Redistribution Merit Share standardises each Delta into a fractional
   share of that game's value. Players who did not play receive WRMS = 0.
3. **Single Game Value (SGV)** — Uniform: total salary pool / total team-games
4. **Season total** — Sum CF across all 82 game slots. Missed games contribute $0,
   so availability is penalised directly in the total — no minimum game filter required.
5. **ROI ratio** — `ROI = Total_CF / Salary`. Values above 1.0 mean the player generated
   more value than he was paid; below 1.0 means he was overpaid relative to production.

### Performance Weights (Table 1, Lautier et al.)

| Variable | Beta  | Description |
|----------|-------|-------------|
| FG2O     | 0.251 | 2-pt FG made |
| FG2X     |-0.349 | 2-pt FG missed |
| FG3O     | 0.537 | 3-pt FG made |
| FG3X     |-0.368 | 3-pt FG missed |
| FTMO     | 0.122 | FT made |
| FTMX     |-0.220 | FT missed |
| PF       |-0.197 | Personal fouls |
| AORB     | 0.356 | Uncontested offensive rebounds |
| ADRB     | 0.316 | Uncontested defensive rebounds |
| STL      | 0.443 | Steals |
| BLK      | 0.132 | Blocks |
| TOV      |-0.347 | Turnovers |
| PFD      | 0.214 | Fouls drawn |
| SAST     | 0.076 | Screen assists |
| CHGD     | 0.522 | Charges drawn |
| AC2P     | 0.041 | 2-pt shots contested (defense) |
| C3P      |-0.067 | 3-pt shots contested (defense) |
| DBOX     | 0.053 | Defensive box outs |
| DFGO     |-0.230 | Defended FG made (opponent scored on you) |
| DFGX     | 0.086 | Defended FG missed (you contested successfully) |
| DDIS     |-1.000 | Defensive distance traveled |
| APM      | 0.016 | Non-assist passes |
| OCRB     | 0.290 | Contested offensive rebounds |
| DCRB     | 0.338 | Contested defensive rebounds |

### Required input files (in `data/raw/`)
| File | Source |
|---|---|
| `nba_game_logs_2024-25.xlsx` | 02_collect_player_stats.ipynb (game-by-game box scores) |
| `nba_advanced_stats_2024-25.xlsx` | 02_collect_player_stats.ipynb (season-level advanced stats) |
| `nba_salaries_2024-25.xlsx` | 01_scrape_salaries.ipynb |

Every formula used below is defined in `src/nba_roi/roi.py` and covered by `tests/test_roi.py`. This notebook walks through the steps one at a time; `python scripts/run_roi.py` runs them all at once.

---
## 0. Imports & Config

In [ ]:
import pandas as pd

from nba_roi import config
from nba_roi.config import BETA, ADVANCED_VARS, GAMES_PER_SEASON
from nba_roi.cleaning import normalize_name, clean_salaries, add_shooting_splits
from nba_roi import roi

print(f"Performance model: Linear weighted score ({len(BETA)} variables, Lautier et al. weights)")
print("ROI method: Total_CF / Salary ratio  (1.0 = break-even)")

---
## 1. Load Data

In [ ]:
logs, adv, sal_raw = roi.load_raw_data()
print(f"Game logs loaded:      {len(logs):,} rows  |  {logs['PLAYER_ID'].nunique()} players")
print(f"Advanced stats loaded: {len(adv)} players")
print(f"Salaries loaded:       {len(sal_raw):,} rows")
sal_raw.head()

---
## 2. Clean & Prepare Data

Names are normalized (accents, ESPN position suffixes, spacing, case) so ESPN salaries can be joined to NBA API players. See `normalize_name` in `src/nba_roi/cleaning.py`.

In [ ]:
sal = clean_salaries(sal_raw, min_salary=config.MIN_SALARY)
print(f"Salaries after cleaning: {len(sal)} players")
sal[['Player', 'PLAYER_NAME_CLEAN', 'SALARY_CLEAN']].head(10)

In [ ]:
# Derive shooting variables from game logs
logs = add_shooting_splits(logs)
logs['PLAYER_NAME_CLEAN'] = logs['PLAYER_NAME'].apply(normalize_name)
missing_adv = [v for v in ADVANCED_VARS if v not in adv.columns]
print('All advanced stat columns present.' if not missing_adv else f'WARNING: missing {missing_adv}')

---
## 3. Compute Per-Game Performance Score (Delta)

The performance score for player m in game g is a weighted sum of centered statistics:

    Delta_gm = sum_j [ beta_j * (x_gm_j - x_bar_j) ]

Where x_gm is the vector of player m's statistics in game g and x_bar is the mean
of each stat across ALL player-game observations. Centering ensures a perfectly
average performance scores zero and the weights encode each action's impact on winning
(from Lautier et al. Table 1) without any probability transformation.

### Variable strategy
- **Box score vars** (FG2O, FG2X, FG3O, FG3X, FTMO, FTMX, PF, STL, BLK, TOV, PFD):
  available per-game from game logs — used directly.
- **Advanced tracking vars** (AORB, ADRB, OCRB, DCRB, DFGO, DFGX, AC2P, C3P, DBOX,
  SAST, CHGD, APM, DDIS): only available as season per-game averages — attached as a
  constant per-game rate for each player across all their games.

In [ ]:
logs_merged = roi.attach_advanced_stats(logs, adv)
delta, col_means = roi.compute_delta(logs_merged, BETA)
logs_merged['DELTA'] = delta

print(f"Centering {len(col_means)} variables.")
print(f"  Mean Delta: {delta.mean():.6f}  (should be ~0)")
print(f"  Std  Delta: {delta.std():.4f}")
print(f"  Min / Max:  {delta.min():.4f} / {delta.max():.4f}")
logs_merged[['PLAYER_NAME', 'GAME_DATE', 'DELTA']].head(10)

> **Note:** the NBA game logs have no PFD (fouls drawn) column, so `attach_advanced_stats` sets PFD to 0 for every game, and its weight has no effect. See *Known limitations* in the README.

---
## 4. Compute WRMS (Wealth Redistribution Merit Share)

WRMS standardises each player's Delta into a fractional share of that game's value:

    WRMS_gm = [ (Delta_gm - Delta_bar) / SD(Delta) ] * (1/m_bar) + (1/m_bar)

A perfectly average player earns exactly 1/m_bar (replacement share). Better-than-average
games earn more; worse-than-average games earn less (and can go negative).
Players who did not play receive WRMS = 0.

In [ ]:
m_bar = roi.average_players_per_game(logs_merged, game_col='Game_ID')
logs_merged['WRMS'], delta_bar, delta_std = roi.compute_wrms(logs_merged['DELTA'], m_bar)

print(f"Avg players/game  (m_bar):   {m_bar:.2f}")
print(f"Replacement share (1/m_bar): {1/m_bar:.4%}")
print(f"WRMS mean: {logs_merged['WRMS'].mean():.6f}  (target: {1/m_bar:.6f})")
print(f"WRMS sum:  {logs_merged['WRMS'].sum():.1f}  (unique games: {logs_merged['Game_ID'].nunique()})")

---
## 5. Single Game Value (SGV) — Uniform Approach

    SGV = Total Salary Pool / Total Team-Games

In [ ]:
total_salary_pool = sal['SALARY_CLEAN'].sum()
SGV = roi.single_game_value(total_salary_pool)
print(f"Total salary pool:  ${total_salary_pool:,.0f}")
print(f"Total team-games:   {config.N_TEAMS * GAMES_PER_SEASON:,}")
print(f"Uniform SGV/game:   ${SGV:,.0f}")

---
## 6. Compute Per-Game Cash Flows

    CF_gm = SGV * WRMS_gm

Bad games produce CF_gm < 0 (consistent with the framework). Missed games are
not in the game logs at all, so they contribute $0 to Total_CF automatically —
no special handling needed. All players with salary data are included regardless
of games played; the 82-game denominator in the ROI ratio does the penalising.

In [ ]:
logs_merged['CF'] = SGV * logs_merged['WRMS']
player_season = roi.summarize_players(logs_merged)
print(f"Players in season summary: {len(player_season)}")
player_season.sort_values('TOTAL_CF', ascending=False).head(10)

---
## 7. Compute ROI Ratio

    ROI = Total_CF / Salary

Total_CF is the sum of all game cash flows across the season. Missed games contribute
$0, so a player who sat out 20 games has a materially lower Total_CF than one who
played all 82 at the same per-game level. No solver or iteration needed.

Interpretation:
- ROI > 1.0  — player generated more value than he was paid (underpaid)
- ROI = 1.0  — break-even; value produced exactly matches salary
- ROI < 1.0  — player was overpaid relative to his production + availability

In [ ]:
# Run the complete model in one call; identical to the steps above
run = roi.run_roi_model(*roi.load_raw_data())
print(f"Players matched to salary: {len(run.player_season)}")
print(f"Players WITHOUT salary (two-way / G-League, dropped): {len(run.unmatched)}")
assert run.logs['CF'].equals(logs_merged['CF'])  # step-by-step == pipeline

---
## 8. Results

In [ ]:
results = run.results
cols = ['PLAYER_NAME', 'Team', 'SALARY_CLEAN', 'GAMES_PLAYED', 'MEAN_DELTA', 'ROI', 'VALUE_DIFF']
print('=== TOP 20 ROI ===')
print(results[cols].head(20).to_string(index=False))
print('\n=== BOTTOM 20 ROI ===')
print(results[cols].tail(20).to_string(index=False))

In [ ]:
p = run.params
print("=== ROI SUMMARY (Linear Model — Total_CF / Salary) ===")
print(f"Players analysed:  {len(results)}")
print(f"Mean ROI:          {results['ROI'].mean():.3f}x")
print(f"Median ROI:        {results['ROI'].median():.3f}x")
print(f"Std Dev ROI:       {results['ROI'].std():.3f}")
print(f"Above break-even (ROI > 1.0): {(results['ROI'] > 1.0).sum()} players")
print(f"Below break-even (ROI < 1.0): {(results['ROI'] < 1.0).sum()} players")
print(f"\nReplacement Total_CF (82 games at 1/m_bar): ${p['replacement_cf']:,.0f}")
print(f"Average salary (all rostered):              ${p['avg_salary']:,.0f}")

---
## 9. Save Results to Excel

In [ ]:
path = roi.write_workbook(run)
print(f"Saved: {path.relative_to(config.PROJECT_ROOT)}")
print("  Sheet 1: ROI Results          — one row per player with salary data")
print("  Sheet 2: Model Parameters     — all WRMS/SGV assumptions and method notes")
print("  Sheet 3: Performance Weights  — Lautier et al. Table 1 coefficients")
print("  Sheet 4: Game Logs + WRMS     — per-game performance scores & cash flows")